# Premier League Performance & Results Sustainability Analysis

## Problem Statement

Recent results show how many points a football team has earned, but they may
not fully represent the underlying quality of its performances.

This project investigates whether underlying performance metrics can provide
additional information about the sustainability of Premier League results.

Using five seasons of match-level data, rolling five-match features are
constructed from expected goals (xG), expected goals against (xGA),
possession, shots, shots on target, and recent points.

Two predictive models are compared:

1. A results-only baseline using points earned over the previous five matches.
2. A performance model combining recent points with underlying performance
   indicators.

Both models predict points earned over the following five matches and are
evaluated on a chronologically held-out season.

### Research Questions

1. How predictive are recent results of points earned over the following five matches?
2. Do underlying performance metrics improve predictions beyond recent results alone?
3. Can disagreement between recent results and underlying performance help identify potentially unsustainable runs of form?

## 2. Import Libraries and Load Dataset

In [80]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("matches.csv")

In [81]:
df.head()

,Unnamed: 0,date,time,comp,round,day,venue,result,gf,ga,...,match report,notes,sh,sot,dist,fk,pk,pkatt,season,team
0,0,2020-09-21,20:15 (21:15),Premier League,Matchweek 2,Mon,Away,W,3,1,...,Match Report,NaN,13,8,21.1,2,1,1,2024,Manchester City
1,2,2020-09-27,16:30 (17:30),Premier League,Matchweek 3,Sun,Home,L,2,5,...,Match Report,NaN,16,5,19.8,1,0,0,2024,Manchester City
2,4,2020-10-03,17:30 (18:30),Premier League,Matchweek 4,Sat,Away,D,1,1,...,Match Report,NaN,23,1,18.2,1,0,0,2024,Manchester City
3,5,2020-10-17,17:30 (18:30),Premier League,Matchweek 5,Sat,Home,W,1,0,...,Match Report,NaN,13,5,17.7,0,0,0,2024,Manchester City
4,7,2020-10-24,12:30 (13:30),Premier League,Matchweek 6,Sat,Away,D,1,1,...,Match Report,NaN,14,7,20.9,1,0,0,2024,Manchester City


In [82]:
df.columns.tolist()

['Unnamed: 0',
 'date',
 'time',
 'comp',
 'round',
 'day',
 'venue',
 'result',
 'gf',
 'ga',
 'opponent',
 'xg',
 'xga',
 'poss',
 'attendance',
 'captain',
 'formation',
 'referee',
 'match report',
 'notes',
 'sh',
 'sot',
 'dist',
 'fk',
 'pk',
 'pkatt',
 'season',
 'team']

In [83]:
cols = [
    "date", "venue", "result", "gf", "ga", "opponent", "xg", "xga", "poss", "sh", "sot", "season", "team"
]

data = df[cols].copy()
data.head(10)

,date,venue,result,gf,ga,opponent,xg,xga,poss,sh,sot,season,team
0,2020-09-21,Away,W,3,1,Wolves,1.9,0.6,65,13,8,2024,Manchester City
1,2020-09-27,Home,L,2,5,Leicester City,0.9,2.9,72,16,5,2024,Manchester City
2,2020-10-03,Away,D,1,1,Leeds United,1.2,2.4,49,23,1,2024,Manchester City
3,2020-10-17,Home,W,1,0,Arsenal,1.3,0.9,58,13,5,2024,Manchester City
4,2020-10-24,Away,D,1,1,West Ham,1.0,0.3,69,14,7,2024,Manchester City
5,2020-10-31,Away,W,1,0,Sheffield Utd,1.6,0.5,65,16,8,2024,Manchester City
6,2020-11-08,Home,D,1,1,Liverpool,1.4,1.2,54,6,2,2024,Manchester City
7,2020-11-21,Away,L,0,2,Tottenham,1.4,0.7,66,22,5,2024,Manchester City
8,2020-11-28,Home,W,5,0,Burnley,1.7,0.5,68,19,6,2024,Manchester City
9,2020-12-05,Home,W,2,0,Fulham,2.9,0.3,69,14,4,2024,Manchester City


In [84]:
print(data["season"].unique())
print(data["team"].nunique())

[2024 2023 2022 2021 2020]
26


In [85]:
data.sample(10)

,date,venue,result,gf,ga,opponent,xg,xga,poss,sh,sot,season,team
4264,2019-10-19,Home,D,1,1,Southampton,0.9,1.4,57,3,0,2020,Wolverhampton Wanderers
104,2021-03-15,Away,W,1,0,Wolves,0.9,1.1,47,12,4,2024,Liverpool
2983,2022-01-11,Away,L,1,4,Southampton,0.5,1.4,52,5,4,2022,Brentford
3802,2020-09-26,Away,W,1,0,Burnley,0.6,0.5,47,5,1,2021,Southampton
4221,2019-09-01,Away,D,2,2,Arsenal,2.0,2.4,45,12,8,2020,Tottenham Hotspur
1204,2024-03-11,Home,W,3,2,Newcastle Utd,1.6,0.7,44,12,7,2024,Chelsea
31,2021-04-10,Home,L,1,2,Leeds United,2.1,0.1,71,29,7,2024,Manchester City
3341,2021-05-13,Home,L,2,4,Liverpool,1.9,2.7,53,18,2,2021,Manchester United
3101,2022-02-26,Home,L,0,1,Manchester City,0.5,1.8,33,6,2,2022,Everton
3328,2021-02-06,Home,D,3,3,Everton,1.8,1.5,61,14,5,2021,Manchester United


In [86]:
data["date"] = pd.to_datetime(data["date"])

In [87]:
def get_season(date):
    if date.month >= 8:
        return date.year + 1
    else:
        return date.year

data["season_clean"] = data["date"].apply(get_season)

In [88]:
data.groupby("season_clean")["date"].agg(["min", "max", "count"])

,min,max,count
season_clean,,,
2020,2019-08-09,2020-07-26,988
2021,2020-09-12,2021-05-23,1520
2022,2021-08-13,2022-05-22,760
2023,2022-08-05,2023-05-28,760
2024,2023-08-11,2024-05-19,760


## 3. Load and Understand the Dataset

In [89]:
df = pd.read_csv("matches.csv")
df.head()
df.shape

FileNotFoundError: [Errno 2] No such file or directory: 'matches.csv'

## Data Preprocessing

Before analysis, the dataset is checked for duplicate observations, missing values, incorrect data types, and inconsistent or invalid values.

In [ ]:
data.duplicated().sum()

In [ ]:
data.duplicated(
    subset = ["date", "team", "opponent", "venue"]
).sum()

In [ ]:
data.isnull().sum()

In [ ]:
data.dtypes

In [ ]:
print("Results:", data["result"].unique())
print("Venues:", data["venue"].unique())

In [ ]:
data[["gf","ga","xga","poss","sh","sot"]].describe()

In [ ]:
print("Negative goals:", ((data["gf"] < 0) | (data["ga"] < 0)).sum())

print("Negative xG:", ((data["xg"] < 0) | (data["xga"] < 0)).sum())

print("Invalid possession:", ((data["poss"] < 0) | (data["poss"] > 100)).sum())

print("Negative shots:", (data["sh"] < 0).sum())

print("SOT greater than shots:", (data["sot"] > data["sh"]).sum())

In [ ]:
duplicate_matches = data[
    data.duplicated(
        subset = ["date" , "team", "opponent", "venue"],
        keep = False
    )
]

duplicate_matches.sort_values(
    ["date", "team", "opponent", "venue"]
).head(20)


In [ ]:
data = data.drop(columns=["season"])

In [ ]:
data = data.drop_duplicates(
    subset=["date", "team", "opponent", "venue"],
    keep="first"
).copy()

In [ ]:
data = data.reset_index(drop=True)

In [ ]:
print("Dataset shape:", data.shape)

print(
    "Remaining duplicates:",
    data.duplicated(
        subset=["date", "team", "opponent", "venue"]
    ).sum()
)

print("Missing values:", data.isnull().sum().sum())

### Preprocessing Summary

- No missing values were found in the selected variables.
- No invalid numerical or categorical values were identified.
- Exact duplicate detection initially returned zero duplicates.
- A domain-based duplicate check using date, team, opponent, and venue revealed repeated match records.
- Investigation showed that the duplicated observations contained inconsistent season labels.
- The season variable was reconstructed from the match date, the unreliable original season column was removed, and duplicate match observations were eliminated.
- The final cleaned dataset contains 3,800 team-match observations across five Premier League which varies againts the 4788 in the dataset.seasons.

## 5. Exploratory Data Analysis
This section explore the relationships between underlying match performance metric and actual match outcome before predictive modeling.

In [ ]:
result_summary = data.groupby("result")[
    ["gf", "ga", "xg", "xga", "poss", "sh", "sot"]
].mean().round(2)

result_summary

Winning performances show a clear increase in attacking output compared with draws and losses. Winning teams averaged 1.85 xG, 14.43 shots, and 5.53 shots on target, compared with 1.00 xG, 10.75 shots, and 3.11 shots on target in losses. Possession showed a much smaller difference, increasing from 47.73% in losses to only 52.27% in wins. This suggests that chance creation and shot quality may be more strongly associated with winning than possession alone.

In [ ]:
data["xg_diff"] = data["xg"] - data["xga"]
data ["goal_diff"] = data["gf"] - data["ga"]

In [ ]:
points_map = {
    "W": 3,
    "D": 1,
    "L": 0
}

data["points"] = data["result"].map(points_map)

In [ ]:
data[
    ["team", "opponent", "result", "points",
     "xg", "xga", "xg_diff", "goal_diff"]
].sample(10)

In [ ]:
data.groupby("result")[
    ["xg_diff", "goal_diff"]
].mean().round(2)

In [ ]:
positive_xg_losses = data[
    (data["xg_diff"] > 0) &
    (data["result"] == "L")
]

negative_xg_wins = data[
    (data["xg_diff"] < 0) &
    (data["result"] == "W")
]

print("Losses despite positive xG difference:", len(positive_xg_losses))
print("Wins despite negative xG difference:", len(negative_xg_wins))

In [ ]:
total_losses = (data["result"] == "L").sum()
total_wins = (data["result"] == "W").sum()

print(
    "Percentage of losses with positive xG difference:",
    round(len(positive_xg_losses) / total_losses * 100, 2),
    "%"
)

print(
    "Percentage of wins with negative xG difference:",
    round(len(negative_xg_wins) / total_wins * 100, 2),
    "%"
)

In [ ]:
performance_metrics = [
    "xg",
    "xga",
    "xg_diff",
    "poss",
    "sh",
    "sot"
]

data[performance_metrics].corr().round(2)

In [ ]:
import matplotlib.pyplot as plt

corr = data[performance_metrics].corr()

plt.figure(figsize = (8,6))
plt.imshow(corr, cmap = "coolwarm", vmin = -1, vmax = 1)

plt.colorbar(label = "Correlation")

plt.xticks(
    range(len(performance_metrics)),
    performance_metrics
)
plt.yticks(
    range(len(performance_metrics)),
    performance_metrics
)

for i in range(len(performance_metrics)):
    for j in range(len(performance_metrics)):
        plt.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.title("Correlation Between Performance Metrics")
plt.tight_layout()
plt.show()

## 6. Feature Engineering 

In [ ]:
data = data.sort_values(
    ["team", "date"]
).reset_index(drop=True)

In [ ]:
rolling_metrics = [
    "xg",
    "xga",
    "xg_diff",
    "poss",
    "sh",
    "sot",
    "points"
]

for metric in rolling_metrics:
    data[f"{metric}_last5"] = (
        data.groupby("team")[metric]
            .transform(lambda x: x.shift(1).rolling(5).mean())
    )

In [ ]:
data["points_next5"] = (
    data.groupby("team")["points"]
        .transform(
            lambda x: (
                x.shift(-1)
                 .rolling(5)
                 .sum()
                 .shift(-4)
            )
        )
)

In [ ]:
rolling_columns = [
    "team", "date", "opponent", "result",
    "points", "xg_last5", "xga_last5",
    "xg_diff_last5", "poss_last5",
    "sh_last5", "sot_last5", "points_last5"
]

data[rolling_columns].dropna().head(10)

In [ ]:
data["points_total_last5"] = data["points_last5"] * 5

In [ ]:
data[
    [
        "team",
        "date",
        "points_total_last5",
        "xg_diff_last5",
        "sh_last5",
        "sot_last5",
        "points_next5"
    ]
].dropna().head(15)

In [ ]:
arsenal_check = data[
    (data["team"] == "Arsenal") &
    (data["date"] > "2-19-09-22")
][["date", "opponent","result", "points"]].head(5)

arsenal_check

In [ ]:
arsenal_check["points"].sum()

In [ ]:
data = data.sort_values(
    ["team", "season_clean", "date"]
).reset_index(drop = True)

In [ ]:
rolling_metrics = [
    "xg",
    "xga",
    "xg_diff",
    "poss",
    "sh",
    "sot",
    "points"
]

for metric in rolling_metrics:
    data[f"{metric}_last5"] = (
        data.groupby(["team", "season_clean"])[metric]
            .transform(
                lambda x: x.shift(1).rolling(5).mean()
            )
    )

data["points_total_last5"] = data["points_last5"] * 5

In [ ]:
data["points_next5"] = (
    data.groupby(["team", "season_clean"])["points"]
        .transform(
            lambda x:
                x.shift(-1) +
                x.shift(-2) +
                x.shift(-3) +
                x.shift(-4) +
                x.shift(-5)
        )
)

In [ ]:
data[
    [
        "team",
        "date",
        "points_total_last5",
        "xg_diff_last5",
        "sh_last5",
        "sot_last5",
        "points_next5"
    ]
].dropna().head(15)

In [ ]:
arsenal_check = data[
    (data["team"] == "Arsenal") &
    (data["date"] > "2019-09-22") &
    (data["season_clean"] == 2020)
][["date", "opponent", "result", "points"]].head(5)

arsenal_check

In [ ]:
arsenal_check["points"].sum()

In [ ]:
features = [
    "points_total_last5",
    "xg_last5",
    "xga_last5",
    "xg_diff_last5",
    "poss_last5",
    "sh_last5",
    "sot_last5"
]

target = "points_next5"

In [ ]:
model_data = data[
    ["team", "date", "season_clean"] + features + [target]
].dropna().copy()

In [ ]:
model_data.shape

In [ ]:
model_data.isnull().sum()

## 7. Modeling

In [ ]:
model_data.groupby("season_clean").size()

In [ ]:
X = model_data[["points_total_last5"]]
y = model_data["points_next5"]

In [ ]:
train_data = model_data[
    model_data["season_clean"] < 2024
].copy()

test_data = model_data[
    model_data["season_clean"] == 2024
].copy()

In [ ]:
print("Training observations:", train_data.shape[0])
print("Testing observations:", test_data.shape[0])

In [ ]:
print("Model data:", model_data.shape)

print("\nSeason counts:")
print(model_data["season_clean"].value_counts().sort_index())

print("\nTrain:", train_data.shape)
print("Test:", test_data.shape)

print("\nTrain + Test:", len(train_data) + len(test_data))

In [ ]:
X_train_results = train_data[["points_total_last5"]]
X_test_results = test_data[["points_total_last5"]]

y_train = train_data["points_next5"]
y_test = test_data["points_next5"]

In [ ]:
from sklearn.linear_model import LinearRegression

results_model = LinearRegression()

results_model.fit(X_train_results, y_train)

In [ ]:
results_predictions = results_model.predict(X_test_results)

In [ ]:
print("Intercept:", results_model.intercept_)
print("Coefficient:", results_model.coef_[0])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae_results = mean_absolute_error(
    y_test,
    results_predictions
)

rmse_results = np.sqrt(
    mean_squared_error(y_test, results_predictions)
)

r2_results = r2_score(
    y_test,
    results_predictions
)

print("MAE:", round(mae_results, 3))
print("RMSE:", round(rmse_results, 3))
print("R²:", round(r2_results, 3))

In [ ]:
performance_features = [
    "points_total_last5",
    "xg_last5",
    "xga_last5",
    "poss_last5",
    "sh_last5",
    "sot_last5"
]

In [ ]:
X_train_performance = train_data[performance_features]
X_test_performance = test_data[performance_features]

In [ ]:
y_train = train_data["points_next5"]
y_test = test_data["points_next5"]

In [ ]:
performance_model = LinearRegression()

performance_model.fit(
    X_train_performance,
    y_train
)

In [ ]:
performance_predictions = performance_model.predict(
    X_test_performance
)

In [ ]:
mae_performance = mean_absolute_error(
    y_test,
    performance_predictions
)

rmse_performance = np.sqrt(
    mean_squared_error(y_test, performance_predictions)
)

r2_performance = r2_score(
    y_test,
    performance_predictions
)

print("MAE:", round(mae_performance, 3))
print("RMSE:", round(rmse_performance, 3))
print("R²:", round(r2_performance, 3))

In [ ]:
coefficients = pd.DataFrame({
    "Feature": performance_features,
    "Coefficient": performance_model.coef_
})

coefficients.sort_values(
    "Coefficient",
    ascending=False
)

In [ ]:
underlying_features = [
    "xg_last5",
    "xga_last5",
    "poss_last5",
    "sh_last5",
    "sot_last5"
]

X_train_underlying = train_data[underlying_features]
X_test_underlying = test_data[underlying_features]

underlying_model = LinearRegression()

underlying_model.fit(
    X_train_underlying,
    train_data["points_total_last5"]
)

In [ ]:
test_data["performance_implied_points"] = (
    underlying_model.predict(X_test_underlying)
)

In [ ]:
test_data["results_gap"] = (
    test_data["points_total_last5"]
    - test_data["performance_implied_points"]
)

In [ ]:
test_data[
    [
        "team",
        "date",
        "points_total_last5",
        "performance_implied_points",
        "results_gap",
        "points_next5"
    ]
].sort_values(
    "results_gap",
    ascending=False
).head(10)

## 8. Sustainability Analysis

In [ ]:
correlation = test_data[
    ["results_gap", "points_next5"]
].corr()

correlation

In [ ]:
test_data["points_change"] = (
    test_data["points_next5"]
    - test_data["points_total_last5"]
)

In [ ]:
gap_change_corr = test_data[
    ["results_gap", "points_change"]
].corr()

gap_change_corr

The more a team's recent points exceeded what its underlying performance suggested, the more its points tended to decline over the following five matches.

In [ ]:
mae_improvement = (
    (mae_results - mae_performance)
    / mae_results
) * 100

rmse_improvement = (
    (rmse_results - rmse_performance)
    / rmse_results
) * 100

print(
    "MAE improvement:",
    round(mae_improvement, 2),
    "%"
)

print(
    "RMSE improvement:",
    round(rmse_improvement, 2),
    "%"
)

Adding underlying performance metrics reduced MAE by 8.03% and RMSE by 6.51% compared with predicting future five-match points from recent results alone.

## 9. Visualize Results

In [ ]:
model_names = [
    "Results Only",
    "Results + Performance"
]

mae_values = [
    mae_results,
    mae_performance
]

plt.figure(figsize=(8, 5))

bars = plt.bar(
    model_names,
    mae_values
)

plt.ylabel("Mean Absolute Error (Points)")
plt.title("Prediction Error: Results-Only vs Performance Model")

for bar, value in zip(bars, mae_values):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.03,
        f"{value:.2f}",
        ha="center"
    )

plt.tight_layout()
plt.show()

### Model Comparison

The results-only baseline produced an MAE of 2.762 points when predicting
points earned over the following five matches.

Adding underlying performance indicators including xG, xGA, possession,
shots, and shots on target reduced the MAE to 2.540 points, representing
an 8.03% reduction in average prediction error.

RMSE also decreased by 6.51%, while R² increased from 0.117 to 0.228.

These results suggest that underlying performance metrics contain predictive
information about future team results beyond recent points alone.

When recent results are stronger or weaker than the underlying performance suggests, what happens to results afterward?

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    test_data["results_gap"],
    test_data["points_change"],
    alpha=0.5
)

# Regression line
m, b = np.polyfit(
    test_data["results_gap"],
    test_data["points_change"],
    1
)

x_line = np.linspace(
    test_data["results_gap"].min(),
    test_data["results_gap"].max(),
    100
)

plt.plot(
    x_line,
    m * x_line + b
)

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)

plt.xlabel("Recent Results Gap (Actual Points - Performance-Implied Points)")
plt.ylabel("Change in Points Over Next 5 Matches")
plt.title("Recent Results Gap vs Future Points Change")

plt.tight_layout()
plt.show()

### Results Sustainability

The relationship between the recent results gap and subsequent points change
was negative (r = -0.539).

Teams whose recent point totals substantially exceeded the level associated
with their underlying performance tended to experience declines in points
over the following five matches. Conversely, teams whose results lagged their
underlying performance often improved.

Because recent points contribute mathematically to both the results gap and
the subsequent change measure, this relationship should not be interpreted
as independent causal evidence. However, it provides descriptive support for
the pattern observed in the out-of-sample model comparison.

In [ ]:
top_overperformers = (
    test_data
    .sort_values("results_gap", ascending=False)
    .head(10)
    .copy()
)

top_overperformers[
    [
        "team",
        "date",
        "points_total_last5",
        "performance_implied_points",
        "points_next5"
    ]
]

In [ ]:
labels = (
    top_overperformers["team"]
    + "\n"
    + top_overperformers["date"].dt.strftime("%b %d")
)

x = np.arange(len(top_overperformers))

width = 0.35

plt.figure(figsize=(12, 6))

plt.bar(
    x - width/2,
    top_overperformers["points_total_last5"],
    width,
    label="Previous 5 Points"
)

plt.bar(
    x + width/2,
    top_overperformers["points_next5"],
    width,
    label="Next 5 Points"
)

plt.xticks(
    x,
    labels,
    rotation=45,
    ha="right"
)

plt.ylabel("Points")
plt.title(
    "What Happened After the Largest Positive Performance–Results Gaps?"
)

plt.legend()
plt.tight_layout()
plt.show()

### Historical Examples

Several of the largest positive performance–results gaps in the 2023–24
test season were followed by substantial declines in points over the next
five matches.

These examples illustrate how underlying performance indicators may provide
context that recent results alone do not capture. They are presented as
illustrative cases rather than independent evidence, since the observations
were selected based on having unusually large positive gaps.

## 10. Conclusion

This project investigated whether underlying performance metrics can provide
additional information about the sustainability of Premier League results.

A results-only baseline model used points earned over the previous five
matches to predict points earned over the following five matches. On the
2023–24 test season, this model achieved an MAE of 2.762 points, an RMSE of
3.323, and an R² of 0.117.

A second model incorporated recent points alongside underlying performance
indicators including expected goals (xG), expected goals against (xGA),
possession, shots, and shots on target. This reduced MAE to 2.540 and RMSE
to 3.106 while increasing R² to 0.228.

Overall, including underlying performance reduced average prediction error
by 8.03% and RMSE by 6.51% compared with using recent results alone.

The results suggest that recent points do not capture all available
information about a team's future results. Underlying performance metrics
provided additional predictive value and may therefore help identify periods
where recent results are less likely to continue.

The performance-results gap analysis also showed a negative relationship
(r = -0.539) between the gap and subsequent changes in points. However,
because recent points are mathematically involved in both measures, this
correlation should be treated as descriptive supporting evidence rather than
an independent causal result.

## Limitations

- Five-match windows are relatively short and can be influenced by random
  variation in finishing, injuries, opposition strength, and other factors.

- The models do not explicitly account for opponent quality, home-field
  advantage, player availability, tactical changes, transfers, or managerial
  changes.

- Linear regression assumes relatively simple relationships between the
  predictors and future points. More complex models may capture nonlinear
  interactions between performance indicators.

- Several performance variables, such as shots, shots on target, and xG,
  are correlated, which makes individual regression coefficients difficult
  to interpret as isolated effects.

- The sustainability gap analysis is partly affected by mathematical coupling
  and regression toward the mean, so it should not be interpreted as evidence
  that a particular team "deserved" more or fewer points.

- The model was evaluated on one held-out Premier League season. Testing
  across additional seasons and competitions would provide stronger evidence
  of generalizability.